# LanceDB fundamentals

In [1]:
import lancedb
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base)

In [2]:
import json

with open("animals_text_embeddings.json") as file:
    data = json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [3]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="animals_text")

In [4]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Add more data

In [5]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

In [6]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"



## Create empty table

In [7]:
from lancedb.pydantic import LanceModel


class Article(LanceModel):
    title: str
    content: str


vector_db.create_table("articles", schema=Article, exist_ok=True)

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="articles")

In [8]:
vector_db.table_names()

['animals_text', 'articles', 'jokes']

TODO: add data to articles table

In [9]:
articles_data = [
    {
        "title": "Fiskar",
        "content": """ 
Fiskar (Pisces) är en grupp vattenlevande ryggradsdjur med fenor, som indelas i benfiskar, broskfiskar och käklösa fiskar. De flesta arter andas med gälar och är växelvarma. Undantaget är lungfiskar. Eftersom landryggradsdjuren släktskapsmässigt är en typ av kvastfeniga fiskar men inte klassificeras som fiskar är begreppet "fiskar" parafyletiskt.

Vetenskapen om fiskar kallas iktyologi.
""",
    },
    {
        "title": "krabba",
        "content": """ 
Det finns stora variationer mellan de olika arternas storlek från släktet Pinnotheres som är bara några millimeter stor och den japanska spindelkrabban (Macrocheira kaempferi) som med utsträckta armar når en spann av 3,7 meter. 
Alla krabbor har ett tjockt exoskelett som huvudsakligen består av kalciumkarbonat. 
Liksom hos andra tiofotade kräftdjur finns fem ben på varje sida och det främre paret är utrustad med saxformiga klor (chela).
Hos simkrabbor (Portunidae) liknar det sista benparet paddlar som används för simning.[3]
""",
    },
]

vector_db["articles"].add(articles_data)

## Drop a table

In [10]:
vector_db.drop_table("articles")

In [11]:
vector_db.table_names()

['animals_text', 'jokes']

## Vector search in LanceDB

Flow here:
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create vector embedding of the query
3. send in the query_vector to search for closest documents

If we want to do RAG -> send in closest document to LLM to use as context

In [12]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [13]:
query_vector = [0.5, 0.2, 0.9]

vector_db["animals_text"].search(query_vector).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
3,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
4,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
5,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
7,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
8,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673
9,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]",0.2822


## Embeddings API in LanceDB

- automatically generate embeddings

In [22]:
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [21]:
from lancedb.pydantic import Vector

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField() # type:ignore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="jokes")

In [23]:
import pandas as pd 

with open("jokes.json") as file: 
    jokes_data = json.load(file)

df_jokes = pd.DataFrame(jokes_data)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [24]:
vector_db["jokes"].add(df_jokes)

In [27]:
df_jokes_embeddings = vector_db["jokes"].to_pandas()
df_jokes_embeddings.head()

,joke,embedding
0,Parallel lines have so much in common—it’s sad...,"[-0.0032196045, 0.0011167526, -0.014953613, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.012634277, 0.005268097, -0.09136963, 0.0809..."
2,What do you call a snake that runs your script...,"[0.019165039, 0.044677734, -0.021530151, 0.028..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013122559, 0.002374649, -0.05706787, 0.0608..."


In [28]:
df_jokes_embeddings["embedding"][10] # note dimensions 1024

array([ 0.01750183,  0.02137756, -0.01231384, ...,  0.05526733,
        0.04537964,  0.01043701], shape=(1024,), dtype=float32)

In [29]:
vector_db["jokes"].search("chemistry related joke").to_pandas()

,joke,embedding,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.811271
1,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",1.002683
2,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.016093
3,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.023621
4,The C# compiler walked into a bar. The bartend...,"[0.0038490295, 0.006790161, -0.0769043, 0.0265...",1.115737
5,Never trust an atom—they make up everything.,"[-0.008956909, 0.003440857, -0.020614624, 0.04...",1.124001
6,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005886078, 0.020553589, -0.0031318665, 0.01...",1.138077
7,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343...",1.168538
8,I put “root beer” in a square glass… now it’s ...,"[0.008476257, 0.026138306, 0.046020508, 0.0015...",1.171756
9,My math teacher said I’m average… how mean!,"[-0.0026016235, 0.03112793, 0.013343811, 0.030...",1.245639
